In [ ]:
from __future__ import (absolute_import, division,
                        print_function, unicode_literals)

import warnings
warnings.simplefilter('ignore')

# general purpose packages
import pandas as pd
import numpy as np
import os
import json
import time
import re
import csv
import subprocess
import sys

import scipy.stats as stats
import statsmodels.stats as smstats
from statsmodels.stats.multitest import multipletests

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from dotenv import load_dotenv
from pathlib import Path

from multiprocessing import Process, Manager, Pool
import multiprocessing
from functools import partial

from collections import Counter

import seaborn as sns; sns.set()

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
matplotlib.rcParams['backend'] = "Qt5Agg"
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter

from IPython.display import display, Image

from adjustText import adjust_text
import builtins
%matplotlib inline

# for normalization
from sklearn.linear_model import QuantileRegressor

# for survival analysis
import sklearn
from sklearn import set_config

from statsmodels.regression.quantile_regression import QuantReg

# for working with yaml files
import ruamel.yaml

import itertools

# for working with .toml
import tomli_w

%load_ext autoreload
%reload_ext autoreload
%autoreload 2
# this is important to be able to re-import the module after making modifications to the zavolab_pyutils code on Scicore

In [ ]:
def get_pvalue_star(pval, thr=0.05):
    if thr == 0.05:
        if pval < 0.001:
            return "***"
        elif pval < 0.01:
            return "**"
        elif pval < 0.05:
            return "*"
        else:
            return ""
    elif thr == 0.1:
        if pval < 0.001:
            return "***"
        elif pval < 0.01:
            return "**"
        elif pval < 0.1:
            return "*"
        else:
            return ""

In [ ]:
# 1. Load the environment variables
load_dotenv("ONT_seq_PAIP1_Hondele.scicore.env",override=True)

# 2. Reconstruct the subdirs dictionary
subdirs = {
    "lab_group_dir": os.getenv("LAB_GROUP_DIR"),
    "raw_sequencing_data_dir": os.getenv("RAW_SEQUENCING_DATA_DIR"),
    "main_project_dir": os.getenv("MAIN_PROJECT_DIR"),
    "wf_dir": os.getenv("WF_DIR"),
    "human_annotation_dir": os.getenv("HUMAN_ANNOTATION_DIR"),
    "drosophila_annotation_dir": os.getenv("DROSOPHILA_ANNOTATION_DIR"),
    "shared_project_dir": os.getenv("SHARED_PROJECT_DIR"),
    "temp_dir": os.getenv("TEMP_DIR"),
    "slurm_dir": os.getenv("SLURM_DIR"),
    "slurm_scripts_dir": os.getenv("SLURM_SCRIPTS_DIR"),
    "figures_dir": os.getenv("FIGURES_DIR"),
    "tables_dir": os.getenv("TABLES_DIR"),
    "fastq_dir": os.getenv("FASTQ_DIR"),
    "metadata_dir": os.getenv("METADATA_DIR"),
    "wf_runs_dir": os.getenv("WF_RUNS_DIR"),
    "configs_dir": os.getenv("CONFIGS_DIR"),
    "pod5_dir": os.getenv("POD5_DIR"), # nanopore-specific
    "dorado_models_dir": os.getenv("DORADO_MODELS_DIR"), # nanopore-specific
    "nanoflowz_dir": os.getenv("NANOFLOWZ_DIR"), # nanopore-specific
}

# 3. Reconstruct the file_paths dictionary
file_paths = {
    "human_genome_file": os.getenv("HUMAN_GENOME_FILE"),
    "human_annotation_file": os.getenv("HUMAN_ANNOTATION_FILE"),
    "human_basic_annotation_file": os.getenv("HUMAN_BASIC_ANNOTATION_FILE"),
    "human_polyAsite_atlas": os.getenv("HUMAN_POLYASITE_ATLAS"),
    "drosophila_genome_file": os.getenv("DROSOPHILA_GENOME_FILE"),
    "drosophila_annotation_file": os.getenv("DROSOPHILA_ANNOTATION_FILE"),
    "dorado_executor": os.getenv("DORADO_EXECUTOR"), # nanopore-specific
}

# 4. Safely create all subdirectories
# Using os.makedirs is highly preferred over os.system('mkdir -p')
# because it avoids opening a subshell and handles permissions gracefully in pure Python.
for path in subdirs.values():
    if path:  # Safety check to ensure the variable was actually found in the .env
        os.makedirs(path, exist_ok=True)

print("Environment loaded and directories verified.")

# Make custome genome file with transfected construct included

In [ ]:
WF_version = 'v1p4_ONT_PAIP1_human_v1'

dir_path = subdirs['wf_runs_dir']+WF_version+'/'
(Path(dir_path)).mkdir(parents=True, exist_ok=True) # create subdirectory for this version

In [ ]:
ref_genome_dir_path = os.path.join(dir_path, 'reference_genome')
(Path(ref_genome_dir_path)).mkdir(parents=True, exist_ok=True) # create subdirectory
# we've manually copied .gb file "pcDNA5-PAIP1-mEGFP.gb" there

In [ ]:
from zavolab_pyutils import (
    genbank_to_fasta_and_gtf,
)

In [ ]:
input_gb_file_path=os.path.join(ref_genome_dir_path, 'pcDNA5-PAIP1-mEGFP.gb')
out_fasta_file_path=os.path.join(ref_genome_dir_path, 'pcDNA5-PAIP1-mEGFP.fasta')
out_gtf_file_path=os.path.join(ref_genome_dir_path, 'pcDNA5-PAIP1-mEGFP.gtf')

genbank_to_fasta_and_gtf(
    input_gb_file_path=input_gb_file_path,
    out_fasta_file_path=out_fasta_file_path,
    out_gtf_file_path=out_gtf_file_path,
    chromosome_name='pcDNA5-PAIP1-mEGFP',
)

# execute the following in command line to create a custom reference genome file
command = 'cat '+file_paths['human_genome_file']+' '+out_fasta_file_path+' > '+os.path.join(ref_genome_dir_path, 'genome_with_construct.fasta')
print(command)

In [ ]:
gtf = pd.read_csv(file_paths['human_basic_annotation_file'],delimiter="\t",index_col=None,header=None,skiprows=5)

In [ ]:
construct_gtf = pd.read_csv(out_gtf_file_path,delimiter="\t",index_col=None,header=None,skiprows=0)
construct_gtf_sel = construct_gtf.loc[construct_gtf[2]=='CDS'].reset_index(drop=True) # we will only look at CDS elements and make genes and exons out of them, for consistency with GENCODE annotation
construct_gtf_sel['gene_id'] = construct_gtf_sel[8].str.split("label"+' "',expand=True)[1].str.split('"',expand=True)[0]
construct_gtf_sel['gene_id'] = construct_gtf_sel.apply(lambda x:x['gene_id'].replace(' ','_')+'_gene',1)
construct_gtf_sel = construct_gtf_sel.loc[~((construct_gtf_sel[3]==3872)&(construct_gtf_sel[4]==3874))].reset_index(drop=True) # discard weird element of EGFP
construct_gtf_sel['gene_name'] = construct_gtf_sel['gene_id']
construct_gtf_sel['gene_type'] = "protein_coding"

construct_gtf_sel[2] = 'gene'
construct_gtf_sel[8] = 'gene_id "'+construct_gtf_sel['gene_id']+'"; gene_type "'+construct_gtf_sel['gene_type']+'"; gene_name "'+construct_gtf_sel['gene_name']+'";'
construct_gtf_sel_genes = construct_gtf_sel.copy()
construct_gtf_sel_genes['order'] = 1

construct_gtf_sel[2] = 'transcript'
construct_gtf_sel[8] = 'gene_id "'+construct_gtf_sel['gene_id']+'"; transcript_id "'+construct_gtf_sel['gene_id']+'.tr'+\
                        '"; gene_type "'+construct_gtf_sel['gene_type']+'"; gene_name "'+construct_gtf_sel['gene_name']+'"; transcript_type "'+construct_gtf_sel['gene_type']+\
                        '"; transcript_name "'+construct_gtf_sel['gene_id']+'.tr'+'"; transcript_support_level "1";'
construct_gtf_sel_transcripts = construct_gtf_sel.copy()
construct_gtf_sel_transcripts['order'] = 2

construct_gtf_sel[2] = 'exon'
construct_gtf_sel[8] = 'gene_id "'+construct_gtf_sel[0]+'"; transcript_id "'+construct_gtf_sel[0]+'.tr'+'"; gene_type "'+construct_gtf_sel['gene_type']+'"; gene_name "'+construct_gtf_sel[0]+\
                        '"; transcript_type "'+construct_gtf_sel['gene_type']+'"; transcript_name "'+construct_gtf_sel[0]+'.tr'+'"; exon_number 1; exon_id "'+construct_gtf_sel[0]+'.ex'+'"; transcript_support_level "1";'
construct_gtf_sel_exons = construct_gtf_sel.copy()
construct_gtf_sel_exons['order'] = 3

construct_gtf_sel_final = pd.concat([construct_gtf_sel_genes,construct_gtf_sel_transcripts,construct_gtf_sel_exons]).sort_values([0,'order']).drop(['order'],axis=1)
construct_gtf_sel_final[1]='CUSTOM'

In [ ]:
gtf_df_compiled = pd.concat([gtf,construct_gtf_sel_final[range(0,9)]]).reset_index(drop=True) # appended constructs

In [ ]:
gtf_df_compiled.to_csv(os.path.join(ref_genome_dir_path, 'compiled_genome_annotation.gtf'), sep=str('\t'),header=False,index=None,quoting=csv.QUOTE_NONE)

# Prepare start samples

In [ ]:
paths_to_look = [
subdirs['pod5_dir']+'p41170_o41323_1/pod5_pass/',
]

pod5_file_paths = []
for path_ in paths_to_look:
    command = """find """+path_+""" -name '*.pod5' > """+subdirs['temp_dir']+"""pod5_files.tsv"""
    out = subprocess.check_output(command, shell=True)
    tmp = pd.read_csv(subdirs['temp_dir']+'pod5_files.tsv',delimiter="\t",
                                   index_col=None,header=None)
    pod5_file_paths.append(tmp)
pod5_file_paths = pd.concat(pod5_file_paths).reset_index(drop=True)
pod5_file_paths.columns = ['pod5']
pod5_file_paths['sample_id'] = pod5_file_paths.apply(lambda x:x['pod5'].split('/')[-2],1)

# we are only interested in barcodes 01-12
sel_sample_ids = range(1,13)
sel_sample_ids = [('barcode0'+str(elem) if len(str(elem))==1 else 'barcode'+str(elem)) for elem in sel_sample_ids]
pod5_file_paths = pod5_file_paths.loc[pod5_file_paths['sample_id'].isin(sel_sample_ids)].reset_index(drop=True)

We will run nanoflowz independently for Hela (human cell line) and Drosophila samples

In [ ]:
# load sample metadata
barcodes_metadata_df = pd.read_csv(subdirs['metadata_dir']+'barcodes_metadata.tsv',delimiter="\t",
                                   index_col=None,header=0)
barcodes_metadata_df['sample_id'] = barcodes_metadata_df.apply(lambda x:'barcode'+('0' if x['Barcode number']<10 else '')+str(x['Barcode number']),1)
barcodes_metadata_df['organism'] = barcodes_metadata_df.apply(lambda x:'human' if ('human' in x['species'].lower()) else 'drosophila',1)

In [ ]:
barcodes_metadata_df

In [ ]:
pod5_file_paths = pd.merge(pod5_file_paths,barcodes_metadata_df[['sample_id','organism']],how='left',on='sample_id')

In [ ]:
# specify outdir and input samples .tsv for nanoflowz runs

organisms = ['human','drosophila']

for organism in organisms:
    WF_version = 'v1p4_ONT_PAIP1_'+organism+'_v1' # v1p4 at the beginning indicated Dorado version, while v1 at the end indicates the particular version of the analysis
    
    dir_path = subdirs['wf_runs_dir']+WF_version+'/'
    (Path(dir_path)).mkdir(parents=True, exist_ok=True) # create subdirectory for this version

    pod5_file_paths_cur = pod5_file_paths.loc[pod5_file_paths['organism']==organism].reset_index(drop=True)

    pod5_file_paths_cur[['sample_id','pod5']].to_csv(dir_path+'start_samples.tsv', sep=str('\t'),header=True,index=None,quoting=csv.QUOTE_NONE)

# Configuring nanoflowz execution

We've created a conda env "nextflow" to execute `nanoflowz`:

```bash
conda create --name nextflow bioconda::nextflow
conda activate nextflow
```

In [ ]:
# We installed `dorado` from ONT's github and got the path to executor:
print(file_paths['dorado_executor'])

In [ ]:
# We installed `nanoflowz` from github to this directory:
print(subdirs['nanoflowz_dir'])

In [ ]:
organisms = ['human','drosophila']

for organism in organisms:
    gpu_to_use = ('l40s' if organism=='human' else 'rtx4090')
    
    WF_version = 'v1p4_ONT_PAIP1_'+organism+'_v1'
    dir_path = subdirs['wf_runs_dir']+WF_version+'/'
    
    run_dir = subdirs['wf_runs_dir']+WF_version+'/'
    
    # create a .toml file for polyA tail profiling
    polyA_toml_output_path = os.path.join(dir_path,'polyA_config.toml')
    (Path(polyA_toml_output_path).parent).mkdir(parents=True, exist_ok=True) # create subdir, just in case
    
    config_data = {
        "tail": {
            "tail_interrupt_length": 1
        }
    }
    with open(polyA_toml_output_path, "wb") as f:
        tomli_w.dump(config_data, f)
    
    # define pipeline configuration
    json_output_path = dir_path+'run_params.json'
    json_file_dir = Path(json_output_path).parent
    json_file_dir.mkdir(parents=True, exist_ok=True)
    
    # define custom parameters for nanoflowz run
    ref_genome_dir_path = os.path.join(run_dir, 'reference_genome')
    
    json_params_content = {
        "tsv": str(Path(dir_path+'start_samples.tsv').resolve()),
        "rundir": str(Path(run_dir).resolve()),
        "dorado": str(file_paths['dorado_executor']),
        "model": str(os.path.join(subdirs['dorado_models_dir'], 'dna_r10.4.1_e8.2_400bps_sup@v5.2.0')),
        "polyA": str(polyA_toml_output_path),
        "ref": str(os.path.join(ref_genome_dir_path, 'genome_with_construct.fasta') if organism=='human' else file_paths[organism+'_genome_file']), # for human samples, we'll try with CUSTOM file
        "reference_gtf": str(os.path.join(ref_genome_dir_path, 'compiled_genome_annotation.gtf') if organism=='human' else file_paths[organism+'_annotation_file']), # for human samples, we'll try with CUSTOM file
        "gpu_partition": gpu_to_use, # we'll submit jobs to different gpus for increased efficiency
        "gpu_qos": gpu_to_use+'-30min',
        "gpu_time": '30min',
    }
        
    params_file = Path(json_output_path)
        
    with open(params_file, "w") as f:
        json.dump(json_params_content, f, indent=4)

    # run the printed command from any directory under the LOGIN node, under `nextflow` conda environment, created above
    # note that it also includes '-resume' argument to make sure that by default it won't be executed from the beginning
    cmd = 'nextflow run '+\
    subdirs['nanoflowz_dir']+'main.nf'+\
    ' -params-file '+str(params_file.resolve())+\
    ' -profile conda'+\
    ' -resume'
    print(cmd)

# Define metadata labels and colors uniformly

In [ ]:
def get_metadata_with_labels_and_colors(input_df):
    sel_files = input_df.copy()
    condiction_dict = {
        1:'KD PAIP1',
        2:'KD PAIP1',
        3:'CTRL',
        4:'CTRL',
        5:'OE PAIP1',
        6:'OE PAIP1',
        7:'E1–3',
        8:'E1–3',
        9:'E1–3',
        10:'C1–3',
        11:'C1–3',
        12:'C1–3'
    }
    sel_files["condition"] = sel_files['Barcode number'].map(condiction_dict)
    replicate_dict = {
        1:'R1',
        2:'R2',
        3:'R1',
        4:'R2',
        5:'R1',
        6:'R2',
        7:'R1',
        8:'R2',
        9:'R3',
        10:'R1',
        11:'R2',
        12:'R3'
    }
    sel_files["replicate"] = sel_files['Barcode number'].map(replicate_dict)

    sel_files["sample_label_within_experiment"] = (
        sel_files["condition"].astype('str')+ ";"
        + sel_files["replicate"].astype('str')
    )

    cat = "condition"
    cat_order = ["KD PAIP1", "CTRL", "OE PAIP1","E1–3","C1–3"]
    cat_palette = [
        "orangered",
        "grey",
        "royalblue",
        "green",
        "orange",
    ]
    cat_color_dict = {}
    for k, cat_val in enumerate(cat_order):
        cat_color_dict[cat_val] = cat_palette[k]
    sel_files[cat + ";color"] = sel_files[cat].map(cat_color_dict)

    sel_files = sel_files.sort_values(
        ["species", "condition", "replicate", "sample"]
    ).reset_index(drop=True)
    return sel_files, cat_order, cat_palette

# PolyA-tail lengths vs abundance vs CPA analysis - on gene level

## Hela experiments

In [ ]:
WF_version = "v1p4_ONT_PAIP1_human_v1"
organism = "human"

command = 'find '+os.path.join(subdirs['wf_runs_dir'],WF_version,'results','read_tag_tables')+\
        " -name '*.tags.tsv.gz' > "+\
          os.path.join(subdirs['temp_dir'],"read_tag_tables.files."+WF_version+".txt")
out = subprocess.check_output(command, shell=True)

read_tag_tables_files = pd.read_csv(os.path.join(subdirs['temp_dir'],"read_tag_tables.files."+WF_version+".txt"),delimiter="\t",
                                   index_col=None,header=None)

entity_col = "sample" # should be a unique index for the metadata table
read_tag_tables_files[entity_col] = read_tag_tables_files.apply(lambda x:x[0].split('/')[-1].replace('.tags.tsv.gz',''),1)
read_tag_tables_files = read_tag_tables_files.rename(columns={0:'path'})

# load sample metadata
barcodes_metadata_df = pd.read_csv(subdirs['metadata_dir']+'barcodes_metadata.tsv',delimiter="\t",
                                   index_col=None,header=0)
barcodes_metadata_df['sample_id'] = barcodes_metadata_df.apply(lambda x:'barcode'+('0' if x['Barcode number']<10 else '')+str(x['Barcode number']),1)
barcodes_metadata_df['organism'] = barcodes_metadata_df.apply(lambda x:'human' if ('human' in x['species'].lower()) else 'drosophila',1)

barcodes_metadata_df['sample'] = barcodes_metadata_df['sample_id'] # for consistency with downstream processing

cat_name = "condition"
metadata_df, cat_order, cat_palette = get_metadata_with_labels_and_colors(barcodes_metadata_df.copy())
metadata_df = metadata_df[[entity_col,'condition','replicate','condition;color']].drop_duplicates().reset_index(drop=True)
metadata_df = pd.merge(metadata_df,read_tag_tables_files,how='inner',on=entity_col)

In [ ]:
# select chromosomes at which we want to look
chromosomes_pd = pd.read_csv(os.path.join(subdirs['wf_runs_dir'],WF_version,'reference_genome','compiled_genome_annotation.gtf'),delimiter="\t",
                                   index_col=None,header=None,usecols=[0])
chromosomes_pd = chromosomes_pd.drop_duplicates().reset_index(drop=True)

sel_chromosome_list = list(chromosomes_pd.loc[chromosomes_pd[0].str.startswith('chr')][0])

def natural_sort_key(s):
    parsed = re.split('([0-9]+)', str(s))
    return [(0, int(text)) if text.isdigit() else (1, text.lower()) for text in parsed]

sel_chromosome_list = sorted(sel_chromosome_list, key=natural_sort_key)

In [ ]:
i=0
res_dict = {}

for index,row in metadata_df.iterrows():
    tmp = pd.read_csv(row['path'],delimiter="\t",index_col=None,header=0,usecols=[1,2,3,4])
    cols = list(tmp.columns)

    tmp = tmp.loc[tmp['chromosome'].isin(sel_chromosome_list)].reset_index(drop=True)
    tmp['chromosome'] = pd.Categorical(tmp['chromosome'], categories=sel_chromosome_list, ordered=True)

    # calculate median-per-gene and abundance
    tmp['w'] = 1/tmp['NH'] # weight
    tmp['w_pt'] = tmp['w']*tmp['pt'] # we will calculate weighted mean

    index_cols = ['chromosome','XT']
    gr = tmp.groupby(index_cols).agg({'w':np.sum,'w_pt':np.sum}).reset_index()
    gr['pt_av'] = gr['w_pt']/gr['w']

    # we'll make a sample matrix for average polyA tail lengths and for abundance
    features = ['pt_av','w']
    for feature in features:
        if i==0:
            res_dict[feature] = gr[index_cols+[feature]].rename(columns={feature:row[entity_col]}).copy()
        else:
            res_dict[feature] = pd.merge(res_dict[feature],gr[index_cols+[feature]].rename(columns={feature:row[entity_col]}),
                                         how='outer',on=index_cols)
    if (i+1)%2==0:
        print(str(i)+' done')
    i=i+1

In [ ]:
sel_metadata = metadata_df.copy()
samples_list = list(sel_metadata['sample'])

In [ ]:
res_dict['w'][samples_list] = np.round(res_dict['w'][samples_list].fillna(0)).astype(int) # make integer counts

### Deseq style analysis of gene expression

In [ ]:
from zavolab_pyutils.read_count_data_analysis import (
    apply_deseq2_normalization
)

norm_cur_res_df, sfs_df = apply_deseq2_normalization(res_dict['w'].copy(),sel_metadata.copy(),pseudocount=1)
norm_cur_res_df = pd.concat([res_dict['w'][index_cols],norm_cur_res_df],axis=1) # add index cols

In [ ]:
subdirs['figures_dir']

In [ ]:
from zavolab_pyutils.visualization import (
    plot_size_factors,
)

plot_size_factors(sfs_df,savefig_path=subdirs['figures_dir']+'gene_expression_analysis/Hela/diagnostic_plots/library_size_vs_SF.png')

In [ ]:
log2_norm_cur_res_df = norm_cur_res_df.copy() # make a log2 version for PCA and may be smth else
log2_norm_cur_res_df[samples_list] = np.log2(log2_norm_cur_res_df[samples_list]) # now we have normalized counts with respect to library size

In [ ]:
from zavolab_pyutils.annotation import (
    parse_gtf_attributes_into_pd_dataframes,
)

gtf_df, genes_df, exons_df = parse_gtf_attributes_into_pd_dataframes(os.path.join(subdirs['wf_runs_dir'],WF_version,'reference_genome','compiled_genome_annotation.gtf'),
                                                                     input_skiprows=0,
                                                                     gene_type_field = "gene_type",
                                                                     extract_exon_number = True, 
                                                                     extract_gene_name_in_exons = True,
                                                                     verbose=False,
                                                                    )

In [ ]:
# Run PCA

from zavolab_pyutils.visualization import (
    pca_plot
)

log2_norm_cur_res_df['m'] = log2_norm_cur_res_df[samples_list].mean(axis=1)
PCA_UMAP_subset = log2_norm_cur_res_df.loc[log2_norm_cur_res_df['m']>log2_norm_cur_res_df['m'].quantile(0.3)].copy().reset_index(drop=True)
print(f"used {len(PCA_UMAP_subset)} genes for PCA analysis")

savefig_path = (
    subdirs['figures_dir']+'gene_expression_analysis/Hela/PCA_gene_expression.Deseq_norm_counts.png'
)

hue_column = cat_name
palette = cat_palette[:3]
hue_order = cat_order[:3]

pca_plot(
    PCA_UMAP_subset,
    samples_list,
    sel_metadata,
    hue_column,
    savefig_path,
    sns_color_palette = palette,
    hue_order = hue_order,
    calculate_permanova_R2=True,
    add_text_labels_for_samples=True,
    text_label_column_for_samples='replicate',
    text_label_size_for_samples=8,
    s_param=40,
    figsize=(4.2,4.2),
)

### Sanity style analysis

In [ ]:
# run additionally gtf parsing in case Deseq part was not used
from zavolab_pyutils.annotation import (
    parse_gtf_attributes_into_pd_dataframes,
)

gtf_df, genes_df, exons_df = parse_gtf_attributes_into_pd_dataframes(os.path.join(subdirs['wf_runs_dir'],WF_version,'reference_genome','compiled_genome_annotation.gtf'),
                                                                     input_skiprows=0,
                                                                     gene_type_field = "gene_type",
                                                                     extract_exon_number = True, 
                                                                     extract_gene_name_in_exons = True,
                                                                     verbose=False,
                                                                    )

In [ ]:
from zavolab_pyutils.read_count_data_analysis import (
    apply_deseq2_normalization, get_MultiDimR2,
    prepare_isoform_sanity_matrix, 
    apply_sanity_normalization_full_bayesian, 
    test_differential_relative_usage,
    test_differential_expression
)

In [ ]:
# we'll exclude overexpression samples from consideration
sel_metadata = metadata_df.loc[metadata_df['condition']!='OE PAIP1'].reset_index(drop=True).copy()
samples_list = list(sel_metadata['sample'])

In [ ]:
# we perform Sanity normalization and variance estimation
raw_counts_df = res_dict['w'].copy()
raw_counts_df.index = raw_counts_df['XT'].values

# Sanity-normalized counts are already log2-transformed
sanity_norm_counts_df, sanity_means_df, sanity_relative_errors_df, sanity_absolute_errors_df, sanity_vg_df, median_lib_size, variances_df = apply_sanity_normalization_full_bayesian(
    counts_df=raw_counts_df, 
    metadata_df=sel_metadata.copy(), 
    sample_col='sample', 
    cond_col='condition',
    vmin=0.0000001, 
    vmax=100,
    n_cores=12,
    empirical_bayes=False, # < Important!
    loess_variance_threshold_q=0.15,
)

In [ ]:
# Diagnostic plots for Sanity outputs

from zavolab_pyutils.visualization import (
    plot_variance_vs_expression,
    plot_mean_vs_cv,
)

# discard the effect from library size, as was done in original Sanity paper
ToCompare_sanity_norm_counts_df = sanity_norm_counts_df-np.log2(median_lib_size)

sanity_vg_df['inferred_v_g'] = sanity_vg_df['MAP_v_g'] # use MAP estimate for Vg to plot, in-line with original Sanity implementation
plot_variance_vs_expression(
    ToCompare_sanity_norm_counts_df, sanity_vg_df,
    savefig_path=subdirs['figures_dir']+'gene_expression_analysis/Hela/pySanity/variance_vs_expr.png',
    true_vg=None, # This is to compare with True value specified during simulation above
    ylim = (0,10),
)

natScale_ToCompare_sanity_norm_counts_df = 2**ToCompare_sanity_norm_counts_df

sanity_plot_data = plot_mean_vs_cv(
natScale_ToCompare_sanity_norm_counts_df, sel_metadata.copy(),
savefig_path=subdirs['figures_dir']+'gene_expression_analysis/Hela/pySanity/cv_vs_mean_plot.png')

In [ ]:
# Run PCA - now based on Sanity normalized counts

from zavolab_pyutils.visualization import (
    pca_plot
)

sanity_norm_counts_df['m'] = sanity_norm_counts_df[samples_list].mean(axis=1)
PCA_UMAP_subset = sanity_norm_counts_df.loc[sanity_norm_counts_df['m']>sanity_norm_counts_df['m'].quantile(0.1)].copy().reset_index(drop=True)
print(len(PCA_UMAP_subset))

savefig_path = (
    subdirs['figures_dir']+'gene_expression_analysis/Hela/PCA_gene_expression.Sanity_norm_counts.png'
)

hue_column = cat_name
palette = cat_palette[:2]
hue_order = cat_order[:2]

pca_plot(
    PCA_UMAP_subset,
    samples_list,
    sel_metadata,
    "condition",
    savefig_path,
    sns_color_palette = palette,
    hue_order = hue_order,
    calculate_permanova_R2=True,
    permanova_R2_ajusted = False, # not adjusted for number of features
    s_param=40,
    figsize=(4.2,4.2),
)

In [ ]:
sanity_DE_df = test_differential_expression(
        means_df=sanity_means_df, 
        errors_df=sanity_relative_errors_df, # do not include global uncertainty to be a bit less conservative
        cond_A="KD PAIP1", # numerator in FC
        cond_B="CTRL"
    )
sanity_DE_df['gene_id'] = sanity_DE_df.index
sanity_DE_df = pd.merge(genes_df[['gene_id','gene_name','gene_type']],sanity_DE_df,how='right',on=['gene_id'])
sanity_DE_df = sanity_DE_df.sort_values('p_value').reset_index(drop=True)

In [ ]:
out_dir_path = os.path.join(subdirs['tables_dir'],'gene_expression_analysis','Hela','pySanity','for_GO')
(Path(out_dir_path)).mkdir(parents=True, exist_ok=True) # create subdirectory

sanity_DE_df.loc[(sanity_DE_df['padj']<0.05)&(sanity_DE_df['log2FC']>0)&(
    ~sanity_DE_df['gene_name'].isna())][['gene_name']].to_csv(
    os.path.join(out_dir_path,'significantly_upregulated.txt'),
    sep=str("\t"),
    header=False,
    index=None,
    quoting=csv.QUOTE_NONE,
)
sanity_DE_df.loc[(sanity_DE_df['padj']<0.05)&(sanity_DE_df['log2FC']<0)&(
    ~sanity_DE_df['gene_name'].isna())][['gene_name']].to_csv(
    os.path.join(out_dir_path,'significantly_downregulated.txt'),
    sep=str("\t"),
    header=False,
    index=None,
    quoting=csv.QUOTE_NONE,
)
sanity_DE_df.loc[(~sanity_DE_df['gene_name'].isna())][['gene_name']].to_csv(
    os.path.join(out_dir_path,'background.txt'),
    sep=str("\t"),
    header=False,
    index=None,
    quoting=csv.QUOTE_NONE,
)

In [ ]:
# plot expression of selected genes across conditions along with its 95% confidence intervals (multiple testing bonferroni-adjusted)
from zavolab_pyutils.visualization import (
    plot_sanity_gene_expression_with_ci,
)

sel_gene_names = ['GAPDH','ACTR1B','PAIP1','DDX19A','DDX19B','PABPC1','PABPN1','MIF4GD','CTIF','NDUFA3','COX6C','MCM3']

# to define the order
sel_genes_df = genes_df.loc[genes_df['gene_name'].isin(sel_gene_names)].reset_index(drop=True).copy()
sel_genes_df['gene_name'] = pd.Categorical(sel_genes_df['gene_name'], categories=sel_gene_names, ordered=True)
sel_genes_df = sel_genes_df.sort_values('gene_name')
selected_genes = list(sel_genes_df['gene_id'])

order = cat_order[:2]
palette = cat_palette[:2]

plot_sanity_gene_expression_with_ci(
    sample_norm_df=sanity_norm_counts_df, 
    means_df=sanity_means_df, 
    errors_df=sanity_relative_errors_df, # we'll use relative errors (a bit smaller than absolute errors)
    metadata_df=sel_metadata.copy(), 
    selected_genes=selected_genes, 
    adjust_multiple_comparisons=True,
    savefig_path=subdirs['figures_dir']+'gene_expression_analysis/Hela/pySanity/over_genes/'+'_'.join(sel_gene_names)+'.expression.png',
    genes_df=genes_df.copy(),
    condition_order = order,
    palette = palette,
    subplot_height = 3.2,
)

In [ ]:
def get_iqr(x):
    iqr = x.quantile(0.75)-x.quantile(0.25)
    return iqr   

x_feature,x_feature_label = 'pt','polyA tail length, nt'
y_feature,y_feature_label = 'sample_label',''
order = list(metadata_df['sample_label'])
palette = list(metadata_df['color'])

comparisons = [(0,1),(2,3),(0,2),(1,3)]

sns.set(font_scale=1)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=True, figsize=(3,2))

data = reads_all_df

ax = sns.boxplot(data = data,x=x_feature,y=y_feature,palette=palette,order=order,showfliers=False)

# prev_point = 0
# for comparison in comparisons:
#     a = data.loc[data[y_feature]==order[comparison[0]]][x_feature]
#     b = data.loc[data[y_feature]==order[comparison[1]]][x_feature]
#     pval = stats.mannwhitneyu(a,b)[1]
#     star = get_pvalue_star(pval)
#     h = 0.2*get_iqr(data[x_feature].astype('float'))
#     BP_top_a = a.quantile(0.75)+1.6*get_iqr(a)
#     BP_top_b = b.quantile(0.75)+1.6*get_iqr(b)
#     start_point = max(BP_top_b,BP_top_a,prev_point)+h*3.5*int(prev_point>0)
#     cat_pos_1 = min(comparison)
#     cat_pos_2 = max(comparison)
#     ax.plot([start_point, start_point+h, start_point+h, start_point],[cat_pos_1, cat_pos_1, cat_pos_2, cat_pos_2], lw=0.5,color='black')
#     ax.text(start_point+1.3*h,0.5*(cat_pos_1+cat_pos_2), star, ha='left', va='center', color='black',size=7)
#     prev_point = start_point
#     break
ax.set(xlabel=x_feature_label,ylabel=y_feature_label,title='ONT reads, all genes')
ax.tick_params(bottom=True)

out = subprocess.check_output('mkdir -p '+subdirs['figures_dir']+'PolyA_tail_lengths/', shell=True)
fig.savefig(subdirs['figures_dir']+'PolyA_tail_lengths/1.boxplots_with_pt.all_genes.png',bbox_inches='tight',dpi=600)
fig.savefig(subdirs['figures_dir']+'PolyA_tail_lengths/1.boxplots_with_pt.all_genes.pdf',bbox_inches='tight',dpi=600)

In [ ]:
# load sample metadata
barcodes_metadata_df = pd.read_csv(subdirs['metadata_dir']+'barcodes_metadata.tsv',delimiter="\t",
                                   index_col=None,header=0)
barcodes_metadata_df['sample_id'] = barcodes_metadata_df.apply(lambda x:'barcode'+('0' if x['Barcode number']<10 else '')+str(x['Barcode number']),1)
barcodes_metadata_df['organism'] = barcodes_metadata_df.apply(lambda x:'human' if ('human' in x['species'].lower()) else 'drosophila',1)

barcodes_metadata_df['sample'] = barcodes_metadata_df['sample_id'] # for consistency with downstream processing